# Pipeline: Resident Health Decline Risk (Next Month)

## Problem framing
**Who cares**: Case management + wellbeing coordinator.

**Business question**: Which residents are at risk of a meaningful decline in **general health score next month**, so staff can intervene.

**Predictive goal**: predict `health_delta_next` (regression on the month-over-month change in general health score). A negative prediction = expected decline.

**Label definition** (default):
- `health_decline_next_month = 1` if next month’s `general_health_score` is at least **0.25 lower** than this month.

## Output
- `predictions_resident_health_decline_next_month.csv`

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

pd.set_option("display.max_columns", 200)

HERE = Path.cwd()
if (HERE / "health_wellbeing_records.csv").exists():
    DATA_DIR = HERE
elif (HERE.parent / "health_wellbeing_records.csv").exists():
    DATA_DIR = HERE.parent
else:
    raise FileNotFoundError("Could not locate health_wellbeing_records.csv from current working directory")

residents = pd.read_csv(DATA_DIR / "residents.csv")
health = pd.read_csv(DATA_DIR / "health_wellbeing_records.csv")
incidents = pd.read_csv(DATA_DIR / "incident_reports.csv")
visits = pd.read_csv(DATA_DIR / "home_visitations.csv")
sessions = pd.read_csv(DATA_DIR / "process_recordings.csv")

health["record_date"] = pd.to_datetime(health["record_date"], errors="coerce")
incidents["incident_date"] = pd.to_datetime(incidents["incident_date"], errors="coerce")
visits["visit_date"] = pd.to_datetime(visits["visit_date"], errors="coerce")
sessions["session_date"] = pd.to_datetime(sessions["session_date"], errors="coerce")

residents.shape, health.shape

((60, 49), (534, 14))

In [2]:
# Build resident-month table for health

hr = health.copy()
hr["month_start"] = hr["record_date"].dt.to_period("M").dt.to_timestamp()

health_rm = hr.groupby(["resident_id", "month_start"]).agg(
    health_general=("general_health_score", "mean"),
    health_nutrition=("nutrition_score", "mean"),
    health_sleep=("sleep_quality_score", "mean"),
    health_energy=("energy_level_score", "mean"),
    bmi=("bmi", "mean"),
    medical_checkups=("medical_checkup_done", lambda s: s.fillna(False).astype(bool).sum()),
    dental_checkups=("dental_checkup_done", lambda s: s.fillna(False).astype(bool).sum()),
    psych_checkups=("psychological_checkup_done", lambda s: s.fillna(False).astype(bool).sum()),
).reset_index()

# Add incident/session/visit counts in the same month
incidents["month_start"] = incidents["incident_date"].dt.to_period("M").dt.to_timestamp()
inc_rm = incidents.groupby(["resident_id", "month_start"]).agg(
    incidents_m=("incident_id", "count"),
    high_sev_m=("severity", lambda s: (s.astype(str).str.lower() == "high").sum()),
).reset_index()

sessions["month_start"] = sessions["session_date"].dt.to_period("M").dt.to_timestamp()
sess_rm = sessions.groupby(["resident_id", "month_start"]).agg(
    sessions_m=("recording_id", "count"),
    session_minutes_m=("session_duration_minutes", "sum"),
    concerns_m=("concerns_flagged", lambda s: s.fillna(False).astype(bool).sum()),
    referrals_m=("referral_made", lambda s: s.fillna(False).astype(bool).sum()),
).reset_index()

visits["month_start"] = visits["visit_date"].dt.to_period("M").dt.to_timestamp()
vis_rm = visits.groupby(["resident_id", "month_start"]).agg(
    visits_m=("visitation_id", "count"),
    safety_concerns_m=("safety_concerns_noted", lambda s: s.fillna(False).astype(bool).sum()),
).reset_index()

rm = health_rm.merge(inc_rm, on=["resident_id", "month_start"], how="left") \
              .merge(sess_rm, on=["resident_id", "month_start"], how="left") \
              .merge(vis_rm, on=["resident_id", "month_start"], how="left")

for c in ["incidents_m", "high_sev_m", "sessions_m", "session_minutes_m", "concerns_m", "referrals_m", "visits_m", "safety_concerns_m"]:
    rm[c] = pd.to_numeric(rm[c], errors="coerce").fillna(0)

# Add static resident context
static_cols = [
    "safehouse_id",
    "case_status",
    "case_category",
    "initial_risk_level",
    "current_risk_level",
    "referral_source",
    "has_special_needs",
    "is_pwd",
]
static_cols = [c for c in static_cols if c in residents.columns]
rm = rm.merge(residents[["resident_id"] + static_cols], on="resident_id", how="left")

rm = rm.sort_values(["resident_id", "month_start"]).reset_index(drop=True)
rm.head()

,resident_id,month_start,health_general,health_nutrition,health_sleep,health_energy,bmi,medical_checkups,dental_checkups,psych_checkups,incidents_m,high_sev_m,sessions_m,session_minutes_m,concerns_m,referrals_m,visits_m,safety_concerns_m,safehouse_id,case_status,case_category,initial_risk_level,current_risk_level,referral_source,has_special_needs,is_pwd
0,1,2023-10-01,3.09,3.02,3.18,2.90,15.5,1,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,Active,Neglected,Critical,High,NGO,True,False
1,1,2023-11-01,3.05,3.07,3.18,2.85,15.6,1,1,1,0.0,0.0,3.0,222.0,2.0,1.0,2.0,1.0,4,Active,Neglected,Critical,High,NGO,True,False
2,1,2023-12-01,3.05,3.21,3.19,2.94,15.6,0,0,0,0.0,0.0,6.0,452.0,0.0,0.0,5.0,0.0,4,Active,Neglected,Critical,High,NGO,True,False
3,1,2024-01-01,3.08,3.27,3.21,2.92,15.4,0,0,0,0.0,0.0,3.0,182.0,1.0,0.0,2.0,2.0,4,Active,Neglected,Critical,High,NGO,True,False
4,1,2024-02-01,3.13,3.30,3.26,2.93,15.6,1,0,1,1.0,0.0,3.0,133.0,0.0,0.0,1.0,0.0,4,Active,Neglected,Critical,High,NGO,True,False


In [3]:
# Target: next-month health score delta (regression)

rm["health_general_next"] = rm.groupby("resident_id")["health_general"].shift(-1)
rm["health_delta_next"] = rm["health_general_next"] - rm["health_general"]

model_df = rm.dropna(subset=["health_general_next", "health_delta_next"]).copy()

TARGET = "health_delta_next"
exclude = {"resident_id", "month_start", "health_general_next", "health_delta_next", "health_decline_next_month"}
feature_cols = [c for c in model_df.columns if c not in exclude]

X_all = model_df[feature_cols].copy()
y_all = model_df[TARGET].astype(float)

X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in numeric_cols]

for c in cat_cols:
    X_train[c] = X_train[c].astype("object")
    X_test[c] = X_test[c].astype("object")

pre = ColumnTransformer(
    [
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ]
)

models = {
    "Ridge": Ridge(alpha=1.0, random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
}

best_name, best_score, best_pipe = None, -1e9, None
for name, model in models.items():
    p = Pipeline([("pre", pre), ("model", model)])
    cv = cross_val_score(p, X_train, y_train, cv=5, scoring="r2")
    print(f"{name}: CV R² = {cv.mean():.3f} ± {cv.std():.3f}")
    if cv.mean() > best_score:
        best_name, best_score, best_pipe = name, cv.mean(), p

print(f"\nBest model: {best_name}")
best_pipe.fit(X_train, y_train)
pipe = best_pipe

pred = pipe.predict(X_test)
print(f"Test MAE: {mean_absolute_error(y_test, pred):.3f} points")
print(f"Test R²:  {r2_score(y_test, pred):.3f}")
print(f"\nTarget stats — mean: {y_all.mean():.3f}, std: {y_all.std():.3f}")

Ridge: CV R² = -0.120 ± 0.059


RandomForest: CV R² = -0.102 ± 0.066

Best model: RandomForest
Test MAE: 0.067 points
Test R²:  -0.102

Target stats — mean: 0.038, std: 0.079


In [4]:
# Feature importance

ohe = pipe.named_steps["pre"].named_transformers_["cat"].named_steps["ohe"] if len(cat_cols) else None
cat_feature_names = ohe.get_feature_names_out(cat_cols).tolist() if ohe is not None else []
feat_names = numeric_cols + cat_feature_names

model = pipe.named_steps["model"]
if hasattr(model, "coef_"):
    imp = model.coef_
    label = "coef"
else:
    imp = model.feature_importances_
    label = "importance"

imp_df = pd.DataFrame({"feature": feat_names, label: imp}).sort_values(label, key=abs, ascending=False)
print(f"Top 20 features by |{label}| ({best_name})")
display(imp_df.head(20))

Top 20 features by |importance| (RandomForest)


,feature,importance
0,health_general,0.122557
2,health_sleep,0.122180
1,health_nutrition,0.104946
3,health_energy,0.103045
4,bmi,0.095384
11,session_minutes_m,0.089571
14,visits_m,0.048499
16,safehouse_id,0.047291
10,sessions_m,0.024264
12,concerns_m,0.019521


In [5]:
# Latest-month predictions per resident (with safehouse context)
safehouses = pd.read_csv(DATA_DIR / "safehouses.csv")

latest = model_df.sort_values(["resident_id", "month_start"]).groupby("resident_id").tail(1).copy()

X_latest = latest[feature_cols].copy()
for c in cat_cols:
    if c in X_latest.columns:
        X_latest[c] = X_latest[c].astype("object")

latest["pred_health_delta_next"] = pipe.predict(X_latest)
latest["high_risk_flag"] = (latest["pred_health_delta_next"] < -0.15).astype(int)

latest = latest.merge(safehouses[["safehouse_id", "safehouse_code", "name"]], on="safehouse_id", how="left")

out_cols = [
    "resident_id", "safehouse_code", "name", "month_start",
    "pred_health_delta_next", "high_risk_flag",
    "health_general", "health_nutrition", "health_sleep", "health_energy",
    "incidents_m", "sessions_m", "visits_m",
    "current_risk_level", "case_status",
]
out_cols = [c for c in out_cols if c in latest.columns]

out = latest[out_cols].copy()
out = out.sort_values("pred_health_delta_next")

out_path = DATA_DIR / "predictions_resident_health_decline_next_month.csv"
out.to_csv(out_path, index=False)

print("Saved:", out_path)
print(f"High-risk residents (predicted delta < -0.15): {out['high_risk_flag'].sum()}")
display(out.head(20))

Saved: /Users/masonzarges/Desktop/INTEX Data/predictions_resident_health_decline_next_month.csv
High-risk residents (predicted delta < -0.15): 0


,resident_id,safehouse_code,name,month_start,pred_health_delta_next,high_risk_flag,health_general,health_nutrition,health_sleep,health_energy,incidents_m,sessions_m,visits_m,current_risk_level,case_status
31,32,SH06,Lighthouse Safehouse 6,2025-09-01,-0.042695,0,3.33,3.54,3.70,2.77,0.0,4.0,1.0,Medium,Active
52,53,SH06,Lighthouse Safehouse 6,2024-08-01,-0.031953,0,3.57,3.64,3.97,3.09,0.0,8.0,1.0,Low,Active
24,25,SH01,Lighthouse Safehouse 1,2024-04-01,-0.030541,0,3.07,3.15,2.97,3.17,0.0,2.0,3.0,High,Active
16,17,SH01,Lighthouse Safehouse 1,2025-08-01,-0.026441,0,3.58,3.13,3.54,2.91,0.0,2.0,1.0,Medium,Active
48,49,SH05,Lighthouse Safehouse 5,2024-10-01,-0.026292,0,3.26,3.24,3.01,3.26,0.0,5.0,1.0,Low,Active
6,7,SH06,Lighthouse Safehouse 6,2025-01-01,-0.017360,0,3.30,3.22,3.21,3.00,0.0,1.0,1.0,Low,Active
42,43,SH02,Lighthouse Safehouse 2,2024-12-01,-0.017153,0,3.47,3.73,3.27,3.04,0.0,0.0,1.0,Low,Closed
44,45,SH03,Lighthouse Safehouse 3,2023-09-01,-0.006197,0,3.48,3.72,3.37,3.33,0.0,5.0,1.0,Medium,Transferred
10,11,SH01,Lighthouse Safehouse 1,2023-11-01,0.002287,0,2.96,3.34,3.35,2.88,0.0,3.0,1.0,Low,Closed
15,16,SH08,Lighthouse Safehouse 8,2023-10-01,0.002907,0,3.24,3.25,3.17,2.77,0.0,2.0,1.0,Low,Active
